In [ ]:
import math
from itertools import combinations

import pandas as pd

In [ ]:
DATA_DIR = "/workspace/data"

subs = pd.read_csv(f"{DATA_DIR}/subscriptions.csv")
act = pd.read_csv(f"{DATA_DIR}/activity.csv")

subs["plan_start"] = pd.to_datetime(subs["plan_start"])
subs["plan_end"] = pd.to_datetime(subs["plan_end"], errors="coerce")
act["event_date"] = pd.to_datetime(act["event_date"])

In [ ]:
reference_date = pd.Timestamp("2025-09-30")
cohort_cutoff = reference_date - pd.Timedelta(days=60)

ENGAGEMENT = {"feature_use", "page_view"}
eng_act = act[act["event_type"].isin(ENGAGEMENT)]

engaged_customers = set(eng_act["customer_id"].unique())
tenured_customers = set(subs.loc[subs["plan_start"] <= cohort_cutoff, "customer_id"])
cohort = engaged_customers & tenured_customers
cohort_size = len(cohort)

In [ ]:
active_customers = set(subs.loc[subs["plan_end"].isna(), "customer_id"])

ended_in_cohort = subs[
    subs["customer_id"].isin(cohort - active_customers) & subs["plan_end"].notna()
].copy()

merged = ended_in_cohort.merge(eng_act, on="customer_id", how="left")
in_period = merged[
    (merged["event_date"] >= merged["plan_start"]) &
    (merged["event_date"] < merged["plan_end"])
]
last_in_period = (
    in_period.groupby("customer_id", as_index=False)["event_date"]
    .max()
    .rename(columns={"event_date": "last_in_period_activity"})
)

df = ended_in_cohort.merge(last_in_period, on="customer_id", how="left")
df["inactivity_gap"] = (df["plan_end"] - df["last_in_period_activity"]).dt.days
df["subscription_duration"] = (df["plan_end"] - df["plan_start"]).dt.days

In [ ]:
long_tenure = df[
    df["last_in_period_activity"].notna() &
    (df["subscription_duration"] >= 90)
]
gap_values = sorted(long_tenure["inactivity_gap"].astype(int).tolist())

if not gap_values:
    inactivity_threshold = 0.0
else:
    n = len(gap_values)
    k = math.ceil(n / 2)
    inactivity_threshold = float(gap_values[k - 1])

In [ ]:
def is_churned(row):
    if pd.isna(row["last_in_period_activity"]):
        return True
    return row["inactivity_gap"] >= inactivity_threshold

df["is_churned"] = df.apply(is_churned, axis=1)
churned_users = int(df["is_churned"].sum())
churn_rate = round(churned_users / cohort_size, 4) if cohort_size else 0.0

In [ ]:
active_subs = subs[subs["plan_end"].isna()].drop_duplicates("customer_id")
sub_start_map = dict(zip(active_subs["customer_id"], active_subs["plan_start"]))

last_activity = (
    eng_act.groupby("customer_id", as_index=False)["event_date"]
    .max()
    .rename(columns={"event_date": "last_event_date"})
)
last_map = dict(zip(last_activity["customer_id"], last_activity["last_event_date"]))

cutoff_7 = reference_date - pd.Timedelta(days=7)

candidates = [
    uid for uid in cohort
    if uid in active_customers
    and pd.notna(last_map.get(uid))
    and last_map.get(uid) <= cutoff_7
]

In [ ]:
labels = {
    uid: {
        f"month:{sub_start_map[uid].to_period('M')}",
        f"dow:{last_map[uid].weekday()}",
    }
    for uid in candidates
}
stale_days = {
    uid: int((reference_date - last_map[uid]).days)
    for uid in candidates
}
target_labels = set().union(*labels.values()) if labels else set()

high_risk_users = []
for size in range(len(candidates) + 1):
    choices = []
    for subset in combinations(sorted(candidates), size):
        covered = set().union(*(labels[uid] for uid in subset)) if subset else set()
        if covered == target_labels:
            min_stale = min(stale_days[uid] for uid in subset) if subset else 0
            choices.append((-min_stale, list(subset)))
    if choices:
        high_risk_users = min(choices)[1]
        break

In [ ]:
cohort_size = int(cohort_size)
churned_users = int(churned_users)
churn_rate = float(churn_rate)
inactivity_threshold = float(inactivity_threshold)
high_risk_users = sorted(high_risk_users)